<a href="https://colab.research.google.com/github/boraevinc/projectlabs/blob/main/Sports_polymarket.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
POLYMARKET PROJECT - SPORTS MARKET LIST BUILDER
================================================
Builds sports.csv: the 500 highest-volume Polymarket Sports markets that closed
between 2026-07-01 and 2026-11-08 (by closedTime), each with its Yes price
7, 3 and 1 day before close and its final outcome (1 = Yes, 0 = No).

HOW TO RUN
    pip install requests pandas
    python polymarket.py            (or whatever you named this file)
Raw downloads are cached in ./raw, so a re-run (or a run after a crash) only
fetches what is missing.

OUTPUTS
    raw/sports_events.json   raw reply from the Gamma API (very large; open it
                             with Python, not a text editor)
    raw/prices/<token>.json  raw price history for each market's Yes token
    sports_working.csv       the final 500 rows + an extra `sport` column
    sports.csv               the same 500 rows in the exact shared format
    sports_report.txt        how many markets were dropped at each filter

PIPELINE (what main() does)
    1. Download closed events tagged Sports (tag id 1), highest volume first.
    2. Flatten to one row per Yes/No market and apply the filters
       (see build_candidates).
    3. For the highest-volume candidates, download hourly price history and
       take the last price at or before close-7d, close-3d and close-1d.
       Drop any market missing one of the three.
    4. Keep the first 500 that pass and write the CSVs and the report.

DESIGN CHOICES (and why) - document these in the project write-up
    - Window by closedTime, never endDate, so the prices cannot leak the answer.
    - Outcome comes from the final outcomePrices (exactly 1 or 0). Anything
      else (cancelled, 50/50) is dropped and counted in the report.
    - Yes/No markets only. Many sports markets use team names as outcomes;
      those have no "Yes" price and are skipped.
    - Events tagged Crypto (21), Tech (1401) or Politics (2) are skipped,
      because the plan gives those categories priority over Sports.
    - A price only counts if it is at most 12 hours older than the target time
      (MAX_STALE_H), so a "7-day price" cannot be several days stale.
    - Markets with under $1,000 of volume are ignored (MIN_VOLUME): their price
      comes from quotes, not trading, so it says little about what people think.
    - DEVIATION FROM THE SHARED PLAN: the plan keeps only events that contain a
      single market. Sports games are multi-market events (win / draw / win,
      spreads, totals), so that rule left only 17 usable markets. Instead we
      rank individual Yes/No markets by volume and keep at most ONE market per
      event (MAX_PER_EVENT), the highest-volume one, so a game is not counted
      two or three times.
    - No per-sport quota: plain top 500 by volume, as in the plan. The sport mix
      is reported afterwards, not designed.

HOW WE GOT HERE (version history)
    v1  Plan: top 500 closed events by volume, one-market events only.
    v2  Idea: spread the 500 across four sports (125 each: football,
        basketball, American football, tennis) for diversification.
    v3  First run: only 17 rows. 19,856 of 20,000 events were multi-market and
        were dropped. -> allowed individual markets, max one per event.
    v4  Second run: 500 rows but 67% football; tennis 4, basketball 22. Short-
        lived game markets are created a day or two before kick-off, so they
        have no 7-day price (5,046 candidates dropped for a missing price).
        American football was padded with $0-volume props (median volume $20).
        -> added MIN_VOLUME.
    v5  Dropped the per-sport quota (QUOTA = 0): plain top 500 by volume.
    Along the way: timeouts now retry instead of crashing, cursor/response
    names fixed to match the API docs (after_cursor / next_cursor / events).

KNOWN LIMITATIONS
    - Result is about 94% football; tennis and basketball are too few
      (4 and 7 markets) to analyse on their own.
    - Sample leans towards markets listed well before close, because the
      7-day price must exist.
    - One market per event is chosen by volume, which measured after the fact
      may favour the favourite's market. Be careful reading a gap between
      prices and outcomes as mispricing without considering this selection.
    - Few long shots (0-20%) and favourites (80-100%) at 7 days, so the
      long-shot tax for Sports will be noisy.
    - Some rows are not game results (e.g. World Cup side markets); they carry
      the Sports tag, so they are kept.
    - Not every API parameter is documented (e.g. allowed values of `order`);
      the script was tested by running it, not only from the docs.
"""
# ---------------------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------------------
import json, os, time, requests, pandas as pd

# Polymarket's two public APIs (no login needed).
# Gamma = events, markets, tags and final outcomes. CLOB = price history.
GAMMA = "https://gamma-api.polymarket.com"
CLOB = "https://clob.polymarket.com"
# Tag id of the Sports category (from the project plan).
SPORTS_TAG_ID = 1
# Date window, applied to each market's closedTime (UTC), inclusive.
START = pd.Timestamp("2026-07-01", tz="UTC")
END = pd.Timestamp("2026-11-08 23:59:59", tz="UTC")
# QUOTA / TOTAL: TOTAL is the number of markets we want. QUOTA is the number to
# take per sport before filling the rest by overall volume (v2 used 125; the
# final version uses 0).
# QUOTA = 0 -> no per-sport quota: plain top-TOTAL markets by volume (matches
# the shared plan). Set e.g. 125 to force a per-sport quota again.
QUOTA, TOTAL = 0, 500
# How old (in hours) a price may be relative to the 7/3/1-day target time.
MAX_STALE_H = 12          # price must be within 12h before the target time
# Safety cap on the event download: 200 pages x 100 events = 20,000 events.
MAX_PAGES = 200           # 100 events/page safety cap
# Ask the server only for events scheduled to end after 1 Jun 2026, so we do not
# page through every sports event ever. The exact filter is still closedTime.
PREFILTER_BY_END_DATE = True
# Rule change vs the shared plan (tell the team): sports games are multi-market
# events, so rank individual Yes/No MARKETS by volume instead of requiring a
# one-market event. MAX_PER_EVENT=1 avoids counting one game 2-3 times
# (home win / draw / away win are dependent). Set ALLOW_MULTI_MARKET=False to
# go back to the strict plan rule.
ALLOW_MULTI_MARKET = True
# Ignore markets with (almost) no trading: their prices are not real signals.
# Set to 0 to disable.
MIN_VOLUME = 1000
MAX_PER_EVENT = 1

# Sport buckets, matched against the event's tag slugs. Extend after the first
# run (unmapped tag slugs are printed). Order = priority if an event has several.
# Used only to LABEL each market with a sport (the `sport` column in
# sports_working.csv). It does not affect which markets are selected when
# QUOTA = 0. Anything not listed here is labelled 'other'.
SPORT_MAP = {
    "football":   ["soccer", "epl", "la-liga", "serie-a", "bundesliga", "ligue-1",
                   "champions-league", "mls", "football"],
    "basketball": ["nba", "basketball", "wnba", "ncaab", "cbb"],
    "american_football": ["nfl", "cfb", "ncaaf", "college-football"],
    "tennis":     ["tennis", "atp", "wta"],
}
# Folder for cached raw downloads.
os.makedirs("raw/prices", exist_ok=True)


# ---------------------------------------------------------------------------
# STEP 1 - DOWNLOAD EVENTS
# ---------------------------------------------------------------------------
# get(): one web request with retries. Timeouts, rate limits (429) and server
# errors (5xx) wait and retry (1, 2, 4... seconds) instead of crashing the run;
# after 6 failed attempts it raises an error.
def get(url, **params):
    for i in range(6):
        try:
            r = requests.get(url, params=params, timeout=60)
        except requests.RequestException:      # timeout / connection reset
            time.sleep(2 ** i); continue
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(2 ** i); continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"gave up on {url}")


# Pages through the closed Sports events, highest volume first (100 per page,
# following the cursor the API returns). The result is saved to
# raw/sports_events.json; if that file exists it is reused, not downloaded again.
def download_events():
    path = "raw/sports_events.json"
    if os.path.exists(path):
        return json.load(open(path))
    events, cursor = [], None
    for _ in range(MAX_PAGES):
        # params/response confirmed against docs.polymarket.com (keyset endpoint)
        p = dict(tag_id=SPORTS_TAG_ID, closed="true", order="volume",
                 ascending="false", limit=100)
        # Loose server-side prefilter so we don't page through all-time events.
        # Uses SCHEDULED end date with a margin; the exact filter is still
        # closedTime, applied client-side. Set to False to disable.
        if PREFILTER_BY_END_DATE:
            p["end_date_min"] = "2026-06-01T00:00:00Z"
        if cursor:
            p["after_cursor"] = cursor
        d = get(f"{GAMMA}/events/keyset", **p)
        batch = d["events"]
        events += batch
        cursor = d.get("next_cursor")   # omitted on the last page
        if not batch or not cursor:
            break
    json.dump(events, open(path, "w"))
    return events


# ---------------------------------------------------------------------------
# STEP 2 - BUILD THE CANDIDATE LIST
# ---------------------------------------------------------------------------
# sport_of(): label an event with a sport from its tag slugs (first match in
# SPORT_MAP wins). Tags that match nothing are counted in `unmapped` so they show
# up in the report, and the event is labelled 'other'.
def sport_of(ev, unmapped):
    slugs = [t.get("slug", "") for t in ev.get("tags", [])]
    for sport, keys in SPORT_MAP.items():
        if any(s in keys for s in slugs):
            return sport
    for s in slugs:
        unmapped[s] = unmapped.get(s, 0) + 1
    return "other"


# jl(): some API fields (outcomes, prices, token ids) arrive as JSON text
# inside a string; this turns them into real lists.
def jl(x):
    return json.loads(x) if isinstance(x, str) else x


# build_candidates(): flatten events into one row per market and apply the filters.
# Every dropped market is counted in `rep` (written to sports_report.txt).
# Filters, in order:
#   1. event also tagged Crypto / Tech / Politics -> skipped (plan priority order)
#   2. outcomes must include 'Yes' and the final prices must be exactly 1 and 0
#   3. closedTime must fall inside START..END
#   4. volume must be at least MIN_VOLUME
#   5. keep only the highest-volume market(s) of each event (MAX_PER_EVENT)
# The result is sorted by volume, highest first.
def build_candidates(events, rep):
    rows, unmapped = [], {}
    rep["events_total"] = len(events)
    rep["dropped_multi_market"] = rep["dropped_not_resolved_yes_no"] = 0
    rep["dropped_outside_window"] = 0
    rep["dropped_higher_priority_category"] = 0
    rep["dropped_extra_markets_per_event"] = 0
    rep["dropped_below_min_volume"] = 0
    HIGHER = {"21", "1401", "2"}   # Crypto, Tech, Politics beat Sports
    for ev in events:
        if {str(t.get("id")) for t in ev.get("tags", [])} & HIGHER:
            rep["dropped_higher_priority_category"] += 1; continue
        mk = ev.get("markets", [])
        if len(mk) > 1 and not ALLOW_MULTI_MARKET:
            rep["dropped_multi_market"] += len(mk); continue
        sport = sport_of(ev, unmapped)
        ev_rows = []
        for m in mk:
            try:
                prices = [float(x) for x in jl(m["outcomePrices"])]
                outs = jl(m["outcomes"]); toks = jl(m["clobTokenIds"])
            except Exception:
                rep["dropped_not_resolved_yes_no"] += 1; continue
            if "Yes" not in outs or sorted(prices) != [0.0, 1.0]:
                rep["dropped_not_resolved_yes_no"] += 1; continue
            iy = outs.index("Yes")
            closed = pd.to_datetime(m.get("closedTime") or ev.get("closedTime"),
                                    utc=True, errors="coerce")
            if pd.isna(closed) or not (START <= closed <= END):
                rep["dropped_outside_window"] += 1; continue
            vol = float(m.get("volume") or m.get("volumeNum") or 0)
            if vol < MIN_VOLUME:
                rep["dropped_below_min_volume"] += 1; continue
            ev_rows.append(dict(slug=m["slug"], sport=sport,
                                volume=vol,
                                closed=closed, token=toks[iy],
                                outcome=int(prices[iy] == 1.0)))
        # keep only the highest-volume market(s) of each event
        ev_rows.sort(key=lambda r: -r["volume"])
        rep["dropped_extra_markets_per_event"] += max(0, len(ev_rows) - MAX_PER_EVENT)
        rows += ev_rows[:MAX_PER_EVENT]
    rep["unmapped_tag_slugs"] = dict(sorted(unmapped.items(), key=lambda kv: -kv[1])[:30])
    return pd.DataFrame(rows).sort_values("volume", ascending=False)


# ---------------------------------------------------------------------------
# STEP 3 - PRICE HISTORY
# ---------------------------------------------------------------------------
# price_history(): hourly prices (fidelity=60 minutes) for a market's Yes token,
# covering the 8 days before close. Cached in raw/prices/<token>.json.
def price_history(token, closed):
    path = f"raw/prices/{token}.json"
    if os.path.exists(path):
        return json.load(open(path))
    start = int((closed - pd.Timedelta(days=8)).timestamp())
    end = int(closed.timestamp())
    d = get(f"{CLOB}/prices-history", market=token, startTs=start,
            endTs=end, fidelity=60)   # params + {history:[{t,p}]} confirmed in docs
    h = d.get("history", [])
    json.dump(h, open(path, "w"))
    return h


# price_at(): the last price at or before time `ts`. Returns None if there is no
# such price, or it is more than MAX_STALE_H hours older than `ts`.
def price_at(hist, ts):
    """Last price at or before ts (unix), no staler than MAX_STALE_H."""
    best = None
    for pt in hist:
        if pt["t"] <= ts:
            best = pt
    if best is None or ts - best["t"] > MAX_STALE_H * 3600:
        return None
    return float(best["p"])


# FETCH_ERRORS: markets whose download kept failing. They are skipped and NOT
# cached, so simply running the script again retries them.
# SEEN: running count used for the progress message.
FETCH_ERRORS = []
SEEN = [0]


# try_market(): get the 7-, 3- and 1-day prices for one candidate. Returns a
# finished row, or None if the download failed or any of the three prices is
# missing (a market needs all three to be kept).
def try_market(r):
    try:
        h = price_history(r.token, r.closed)
    except Exception:                  # keeps failing: skip it, rerun retries it
        FETCH_ERRORS.append(r.slug)
        return None
    SEEN[0] += 1
    if SEEN[0] % 50 == 0:
        print(f"  ...checked {SEEN[0]} markets", flush=True)
    c = int(r.closed.timestamp())
    ps = [price_at(h, c - d * 86400) for d in (7, 3, 1)]
    if any(p is None for p in ps):
        return None
    return dict(slug=r.slug, category="Sports", volume=r.volume,
                close_date=r.closed.strftime("%Y-%m-%d"),
                price_7d=ps[0], price_3d=ps[1], price_1d=ps[2],
                outcome=r.outcome, sport=r.sport)


# ---------------------------------------------------------------------------
# STEP 4 - SELECT THE 500 AND SAVE
# ---------------------------------------------------------------------------
def main():
    rep = {}
    cand = build_candidates(download_events(), rep)
    rep["candidates_by_sport"] = cand.sport.value_counts().to_dict()
    kept, used, no_price = [], set(), 0
    sports = list(SPORT_MAP)
    # pass 1: quota per sport
    for s in sports:
        n = 0
        for r in cand[cand.sport == s].itertuples():
            if n >= QUOTA: break
            row = try_market(r); used.add(r.slug)
            if row: kept.append(row); n += 1
            else: no_price += 1
    rep["after_quota_pass"] = (pd.DataFrame(kept).sport.value_counts().to_dict()
                               if kept else {})
    # With QUOTA = 0 pass 1 takes nothing, so pass 2 does all the work: it walks the
    # candidates from highest volume down and keeps those with all three prices
    # until it has TOTAL (500) rows.
    # pass 2: fill to TOTAL from everything left (incl. "other"), by volume
    for r in cand[~cand.slug.isin(used)].itertuples():
        if len(kept) >= TOTAL: break
        row = try_market(r)
        if row: kept.append(row)
        else: no_price += 1
    rep["dropped_missing_price_point"] = no_price - len(FETCH_ERRORS)
    rep["failed_downloads_rerun_to_retry"] = len(FETCH_ERRORS)
    # Write the results. The assert is a safety net: no duplicate slugs and no
    # blank cells. sports.csv has exactly the shared columns; sports_working.csv
    # also keeps the `sport` column.
    df = pd.DataFrame(kept).sort_values("volume", ascending=False)
    rep["final_rows"] = len(df)
    rep["final_by_sport"] = df.sport.value_counts().to_dict()
    df.to_csv("sports_working.csv", index=False)
    cols = ["slug", "category", "volume", "close_date",
            "price_7d", "price_3d", "price_1d", "outcome"]
    assert df.slug.is_unique and not df[cols].isna().any().any()
    df[cols].to_csv("sports.csv", index=False)
    open("sports_report.txt", "w").write(json.dumps(rep, indent=2, default=str))
    print(json.dumps(rep, indent=2, default=str))


if __name__ == "__main__":
    main()